# 낚시성 뉴스 제목 판별기 학습 (Colab)

이 노트북은 저장소의 스크립트를 순서대로 부르기만 한다. 처리 로직은 모두 `clickbait/` 와 `scripts/` 에 있다.

순서: Drive 연결 → 저장소 받기 → 패키지 설치 → AI Hub 데이터 정제 → 기준선(TF-IDF) → KLUE-BERT 작은 샘플 학습 → KLUE-BERT 전체 학습 → KoBERT 전체 학습 → 평가 → 비교표 → 가장 좋은 모델을 Drive 에 복사.

시작 전에 **런타임 → 런타임 유형 변경 → GPU** 를 고른다.

## 1. Google Drive 연결

AI Hub 원본 ZIP, 체크포인트, 학습한 모델은 모두 Drive 에 둔다. Colab 세션은 언제든 끊길 수 있어서 로컬 디스크(`/content`)에만 둔 것은 사라진다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 설정

아래 값을 팀 환경에 맞게 고친다.

- `REPO_URL`: 팀 저장소 주소. 저장소를 쓰지 않으면 비워 두고 프로젝트 폴더를 `/content/repo` 에 직접 올린다.
- `RAW_ZIP_DIR`: AI Hub 에서 받은 ZIP(`TL_Part1_Clickbait_Auto_EC.zip`, `VL_...` 등)을 넣어 둔 Drive 폴더. 압축은 풀지 않아도 된다.

In [ ]:
REPO_URL = ''  # 예: 'https://github.com/<팀계정>/Clickbait_Detection_for_News_Headlines.git'
REPO_DIR = '/content/repo'
DRIVE_ROOT = '/content/drive/MyDrive/clickbait'
RAW_ZIP_DIR = f'{DRIVE_ROOT}/aihub_raw'
CKPT_ROOT = f'{DRIVE_ROOT}/checkpoints'
MODEL_ROOT = f'{DRIVE_ROOT}/models'
LOCAL_RAW = '/content/aihub_raw'

import os
for d in (CKPT_ROOT, MODEL_ROOT):
    os.makedirs(d, exist_ok=True)

## 3. 저장소 받기와 패키지 설치

이미 받아 둔 저장소가 있으면 최신으로 맞춘다. PyTorch 는 Colab 에 이미 깔려 있다.

In [ ]:
import os
if REPO_URL and not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
elif REPO_URL:
    !git -C {REPO_DIR} pull --ff-only
assert os.path.isdir(REPO_DIR), 'REPO_URL 을 채우거나 프로젝트 폴더를 /content/repo 에 올린다.'
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!nvidia-smi

## 4. 데이터 정제

Drive 에서 바로 읽으면 느려서 Part1 ZIP 만 로컬 디스크로 복사한 뒤 읽는다. Part2 는 본문을 조작한 데이터라 제목만 보는 모델에는 라벨이 잡음이므로 기본으로 쓰지 않는다.

결과: `data/processed/{train,val,test}.csv`, `bodies.jsonl`, `data_report.json`. 같은 명령을 다시 실행해도 같은 결과가 나온다.

In [ ]:
!mkdir -p {LOCAL_RAW}
!cp -n {RAW_ZIP_DIR}/*Part1*.zip {LOCAL_RAW}/
!ls -la {LOCAL_RAW}
!python scripts/prepare_data.py --raw-dir {LOCAL_RAW}

In [ ]:
import json
print(json.dumps(json.load(open('data/processed/data_report.json', encoding='utf-8')), ensure_ascii=False, indent=2))

## 5. 기준선: TF-IDF + 로지스틱 회귀

In [ ]:
!python scripts/train_baseline.py
!python scripts/evaluate.py --model-dir models/baseline --name baseline

## 6. KLUE-BERT 를 작은 샘플로 먼저 학습

전체 학습 전에 학습 데이터 5,000건, 1에폭으로 코드와 GPU 메모리가 괜찮은지 확인한다. 이 결과는 비교표에 넣지 않는다.

In [ ]:
!python scripts/train_bert.py --model-key klue-bert --limit-train 5000 --epochs 1 \
    --output-dir /content/models/klue-bert-sample --checkpoint-dir /content/checkpoints/klue-bert-sample

## 7. KLUE-BERT 전체 학습

세션이 끊기면 위의 1~3번 셀을 다시 실행한 뒤 **이 셀만 다시 실행**한다. `--resume` 이 Drive 의 체크포인트에서 이어 간다(체크포인트가 없으면 처음부터 시작한다).

가장 좋은 모델도 Drive(`MODEL_ROOT`)에 바로 저장한다. 로컬에만 두면 세션이 끊길 때 앞 에폭의 최고 모델이 사라진다.

In [ ]:
!python scripts/train_bert.py --model-key klue-bert --resume \
    --checkpoint-dir {CKPT_ROOT}/klue-bert --output-dir {MODEL_ROOT}/klue-bert
!python scripts/evaluate.py --model-dir {MODEL_ROOT}/klue-bert --name klue-bert --device cuda

## 8. KoBERT 전체 학습

KLUE-BERT 와 같은 설정이다. 끊기면 이 셀만 다시 실행한다.

In [ ]:
!python scripts/train_bert.py --model-key kobert --resume \
    --checkpoint-dir {CKPT_ROOT}/kobert --output-dir {MODEL_ROOT}/kobert
!python scripts/evaluate.py --model-dir {MODEL_ROOT}/kobert --name kobert --device cuda

## 9. 모델 비교

`reports/comparison.md` 를 만든다. 오분류 사례는 `reports/<이름>/misclassified.csv` 에 확신이 큰 오답부터 정렬되어 있다(오류 분석용).

In [ ]:
!python scripts/compare_models.py
from IPython.display import Image, display
for name in ('baseline', 'klue-bert', 'kobert'):
    path = f'reports/{name}/confusion_matrix.png'
    if os.path.exists(path):
        display(Image(path))

## 10. 가장 좋은 모델과 보고서를 Drive 에 복사

비교표를 보고 `BEST` 를 고른다. 서버는 이 폴더를 `models/klue-bert` 같은 위치에 내려받아 쓴다.

In [ ]:
BEST = 'klue-bert'  # 비교표에서 F1 이 가장 높은 BERT 모델 이름
!rm -rf {DRIVE_ROOT}/best_model {DRIVE_ROOT}/reports
!cp -r {MODEL_ROOT}/{BEST} {DRIVE_ROOT}/best_model
!cp -r reports {DRIVE_ROOT}/reports
!ls -la {DRIVE_ROOT}/best_model